# Responses API Overview (Python SDK)

> ⚠️ **Migration Notice:** The [Assistants API](https://platform.openai.com/docs/assistants/overview) has been **deprecated** as of August 26, 2025, and will be **shut down on August 26, 2026**. This notebook demonstrates the recommended replacement: the **Responses API**. See the [official migration guide](https://platform.openai.com/docs/guides/migrate-to-assistants) for full details.

The [Responses API](https://platform.openai.com/docs/api-reference/responses) is OpenAI's next-generation interface for building stateful, agentic experiences. It offers a simpler mental model than the Assistants API while providing the same capabilities — plus new features like deep research, MCP tool support, and computer use.

## Concept mapping: Assistants API → Responses API

| Assistants API | Responses API | Notes |
|---|---|---|
| `Assistant` | `Prompt` (dashboard) | Created in the dashboard, not via API |
| `Thread` | `Conversation` | Server-side state via `client.conversations.create()` |
| `Run` | `Response` | `client.responses.create()` |
| `Run Step` | `Item` | Granular step inside a response |
| `client.beta.threads.runs.create_and_poll()` | `client.responses.create()` | No polling loop needed |

## What we'll cover
1. [Setup](#setup)
2. [Basic text conversation](#basic)
3. [Multi-turn conversation with server-side state](#multi-turn)
4. [Built-in tools: Code Interpreter](#code-interpreter)
5. [Built-in tools: File Search](#file-search)
6. [Function calling](#function-calling)
7. [Streaming](#streaming)
8. [Cleanup](#cleanup)

## 1. Setup <a id='setup'></a>

In [ ]:
%pip install --upgrade openai --quiet

In [ ]:
import json
import os
from IPython.display import display
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "<your OpenAI API key if not set as env var>"))

# Helper to pretty-print response objects
def show_json(obj):
    display(json.loads(obj.model_dump_json()))

# Helper to extract the final text from a response
def get_text(response):
    for item in response.output:
        if item.type == "message":
            for block in item.content:
                if block.type == "output_text":
                    return block.text
    return ""

## 2. Basic text conversation <a id='basic'></a>

The simplest Responses API call. Pass `instructions` (replaces the `Assistant` object's system prompt) and `input` (the user message). No threads, no polling.

In [ ]:
response = client.responses.create(
    model="gpt-4o",
    instructions="You are a personal math tutor. Answer questions briefly, in a sentence or less.",
    input="I need to solve the equation `3x + 11 = 14`. Can you help me?",
)

print(get_text(response))

## 3. Multi-turn conversation with server-side state <a id='multi-turn'></a>

In the Assistants API you created a `Thread` to store conversation history server-side. In the Responses API you have two options:

- **Option A — `previous_response_id`**: chain responses by referencing the last response ID.
- **Option B — `Conversation`**: create a conversation object that groups responses (analogous to a thread).

We'll use both.

### Option A: Chaining via `previous_response_id`

In [ ]:
# Turn 1
r1 = client.responses.create(
    model="gpt-4o",
    instructions="You are a personal math tutor. Answer questions briefly.",
    input="What is the quadratic formula?",
    store=True,  # must be True to use previous_response_id
)
print("Turn 1:", get_text(r1))

# Turn 2 — model automatically has Turn 1 context
r2 = client.responses.create(
    model="gpt-4o",
    instructions="You are a personal math tutor. Answer questions briefly.",
    previous_response_id=r1.id,
    input="Can you give me an example using x² + 5x + 6 = 0?",
    store=True,
)
print("Turn 2:", get_text(r2))

### Option B: Using a Conversation object (analogous to a Thread)

In [ ]:
# Create a conversation (server-side state container)
conversation = client.conversations.create()
print("Conversation ID:", conversation.id)

# Turn 1 — attach to the conversation
r1 = client.responses.create(
    model="gpt-4o",
    instructions="You are a personal math tutor. Answer questions briefly.",
    input="What is the Pythagorean theorem?",
    conversation=conversation.id,
    store=True,
)
print("Turn 1:", get_text(r1))

# Turn 2 — same conversation, history is maintained automatically
r2 = client.responses.create(
    model="gpt-4o",
    instructions="You are a personal math tutor. Answer questions briefly.",
    input="Give me a real-world example of this theorem.",
    conversation=conversation.id,
    store=True,
)
print("Turn 2:", get_text(r2))

## 4. Built-in tools: Code Interpreter <a id='code-interpreter'></a>

Enable Code Interpreter by adding it to the `tools` list. The model can now write and execute Python code in a sandboxed container.

In [ ]:
response = client.responses.create(
    model="gpt-4o",
    instructions="You are a math tutor. Use code to verify your answers.",
    input="Calculate the first 10 Fibonacci numbers and show a bar chart.",
    tools=[
        {
            "type": "code_interpreter",
            "code_interpreter": {"container": {"type": "auto"}}
        }
    ],
)

print(get_text(response))

# Inspect all output items (text, code, images)
for item in response.output:
    print(f"Item type: {item.type}")

## 5. Built-in tools: File Search <a id='file-search'></a>

Upload a file, create a vector store, and use the `file_search` tool to let the model retrieve relevant content from your documents.

In [ ]:
# Upload a file
sample_text = b"""OpenAI was founded in December 2015.
GPT-4 was released in March 2023.
The Responses API was introduced in 2025 as a replacement for the Assistants API.
The Assistants API will be shut down on August 26, 2026.
"""

uploaded_file = client.files.create(
    file=("openai_facts.txt", sample_text, "text/plain"),
    purpose="assistants",
)
print("Uploaded file ID:", uploaded_file.id)

# Create a vector store and add the file
vector_store = client.vector_stores.create(name="OpenAI Facts")
client.vector_stores.files.create(
    vector_store_id=vector_store.id,
    file_id=uploaded_file.id,
)
print("Vector store ID:", vector_store.id)

In [ ]:
# Query the file using the file_search tool
response = client.responses.create(
    model="gpt-4o",
    instructions="Answer questions using only the provided documents.",
    input="When was the Assistants API deprecated and what replaces it?",
    tools=[
        {
            "type": "file_search",
            "file_search": {"vector_store_ids": [vector_store.id]}
        }
    ],
)

print(get_text(response))

In [ ]:
# Cleanup
client.vector_stores.delete(vector_store.id)
client.files.delete(uploaded_file.id)
print("Vector store and file deleted.")

## 6. Function calling <a id='function-calling'></a>

Define custom tools using JSON Schema. When the model decides to call a function, the response will contain a `function_call` item. You execute the function yourself and submit the result back.

In [ ]:
import json

# Define a mock quiz function
def display_quiz(title: str, questions: list) -> str:
    print(f"\n📝 Quiz: {title}")
    answers = []
    for i, q in enumerate(questions):
        print(f"Q{i+1}: {q['question']}")
        if "options" in q:
            for j, opt in enumerate(q["options"]):
                print(f"   {chr(65+j)}) {opt}")
        user_answer = input("Your answer: ")
        answers.append(user_answer)
    return json.dumps({"answers": answers})

# Tool definition (JSON Schema)
quiz_tool = {
    "type": "function",
    "name": "display_quiz",
    "description": "Displays a quiz to the student and returns their answers.",
    "parameters": {
        "type": "object",
        "properties": {
            "title": {"type": "string", "description": "The quiz title."},
            "questions": {
                "type": "array",
                "description": "List of questions.",
                "items": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string"},
                        "options": {"type": "array", "items": {"type": "string"}}
                    },
                    "required": ["question"]
                }
            }
        },
        "required": ["title", "questions"]
    }
}

In [ ]:
def run_quiz_agent(user_message: str) -> str:
    """Agentic loop: handles function calls from the model automatically."""
    messages = [{"role": "user", "content": user_message}]
    previous_response_id = None

    while True:
        kwargs = {
            "model": "gpt-4o",
            "instructions": "You are a math tutor. When asked to quiz a student, use the display_quiz function.",
            "tools": [quiz_tool],
            "store": True,
        }
        if previous_response_id:
            kwargs["previous_response_id"] = previous_response_id
        else:
            kwargs["input"] = messages

        response = client.responses.create(**kwargs)
        previous_response_id = response.id

        # Check for function calls
        function_calls = [
            item for item in response.output if item.type == "function_call"
        ]

        if not function_calls:
            # No function call → final text response
            return get_text(response)

        # Execute each function call and collect results
        tool_results = []
        for fc in function_calls:
            args = json.loads(fc.arguments)
            result = display_quiz(args["title"], args["questions"])
            tool_results.append({
                "type": "function_call_output",
                "call_id": fc.call_id,
                "output": result,
            })

        # Submit tool results back for the next turn
        messages = tool_results


final_response = run_quiz_agent(
    "Give me a short 2-question multiple choice quiz on linear equations."
)
print("\nFinal response:", final_response)

## 7. Streaming <a id='streaming'></a>

The Responses API supports server-sent event streaming. Set `stream=True` and iterate over the returned event stream.

In [ ]:
stream = client.responses.create(
    model="gpt-4o",
    instructions="You are a math tutor. Explain step by step.",
    input="Explain how to complete the square for x² + 6x + 5 = 0.",
    stream=True,
)

print("Streaming response:")
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)

print()  # newline after stream ends

## 8. Cleanup <a id='cleanup'></a>

Unlike the Assistants API, there is no persistent `Assistant` or `Thread` object to delete. Responses are stored automatically (controlled by the `store` parameter). To delete a stored response:

In [ ]:
# Delete a specific response (optional — responses expire automatically)
# client.responses.delete(response_id)

# Delete a conversation
# client.conversations.delete(conversation.id)

print("No persistent Assistant or Thread objects to delete — cleanup is minimal!")

## Summary

| Task | Assistants API (deprecated) | Responses API |
|---|---|---|
| Create assistant | `client.beta.assistants.create()` | Pass `instructions=` directly |
| Create thread | `client.beta.threads.create()` | `client.conversations.create()` |
| Add message | `client.beta.threads.messages.create()` | Pass `input=` directly |
| Run + poll | `client.beta.threads.runs.create_and_poll()` | `client.responses.create()` |
| Continue conversation | Thread ID | `previous_response_id` or `conversation=` |
| Stream | `stream=True` + `on()` event handlers | `stream=True` + async `for event in stream` |
| Function calling | Submit via `runs.submit_tool_outputs()` | Pass `input=[{type: function_call_output}]` |

For more details, see:
- [Responses API reference](https://platform.openai.com/docs/api-reference/responses)
- [Official Assistants → Responses migration guide](https://platform.openai.com/docs/guides/migrate-to-assistants)
- [Conversations API reference](https://platform.openai.com/docs/api-reference/conversations)